Funzione 1

Riceve in input il nome di un pilota e restituisce una lista contenente tre informazioni chiave: - Il totale dei punti accumulati dal pilota durante il campionato. - Il numero di vittorie, ovvero quante volte il pilota è arrivato primo in un Gran Premio. - Il numero di podi, ovvero quante volte il pilota è arrivato tra i primi tre classificati.

Questa funzione sarà utile per analizzare le performance individuali dei piloti e avere una chiara visione delle loro posizioni nel corso della stagione.

In [ ]:
import urllib.request

file_id = "1XOyt1IHEeVg0CrSNblIYMjqcwSh_F8DR"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

urllib.request.urlretrieve(url, "formula1_data.csv")

file_csv = "formula1_data.csv"

import csv    # serve per leggere file CSV

In [ ]:
def analizza_pilota(nome_pilota):
    #Sistema punti
    punti = {1:10, 2:8, 3:6, 4:5, 5:4, 6:3, 7:2, 8:1}

    #Inizializza variabili
    totale_punti = 0
    vittorie = 0
    podi = 0

    #Legge il file CSV
    with open(file_csv, newline='') as file:
        lettore = csv.DictReader(file) #Legge il file CSV e trasforma ogni riga in un dizionario
        for riga in lettore: #Legge csv riga per riga
            pilota = riga["Driver"].strip().lower() #Prende il nome del pilota dalla riga, toglie spazi(strip) e lo rende minuscolo per confronti più precisi (lower)
            posizione = int(riga["Position"])       #Legge la posizione del pilota e la converte in numero intero


            if pilota == nome_pilota.strip().lower():   #Se il pilota della riga corrisponde a quello cercato
                totale_punti += punti.get(posizione, 0)  #Aggiunge ai punti totali quelli corrispondenti alla posizione del pilota (0 se non prende punti)

                #Conta una vittoria se è arrivato primo
                if posizione == 1:
                    vittorie += 1

                #Conta un podio se è arrivato tra i primi tre
                if 1 <= posizione <= 3:
                    podi += 1

    #Ritorna la lista richiesta
    return [totale_punti, vittorie, podi]

In [ ]:
#Apre il file CSV per estrarre tutti i nomi dei piloti
with open(file_csv, newline='') as file:
    lettore = csv.DictReader(file)
    piloti = set()  #uso un set per evitare duplicati

    #Aggiunge ogni pilota trovato nel file al set
    for riga in lettore:
        piloti.add(riga["Driver"].strip())

In [ ]:
#Stampa l’elenco dei piloti disponibili
print("--- PILOTI DISPONIBILI ---")
for nome in sorted(piloti): #Ordina alfabeticamente i nomi
    print("-", nome)

nome = input("\nInserisci il nome del pilota da analizzare: ")
risultato = analizza_pilota(nome)

--- PILOTI DISPONIBILI ---
- Alonso
- Glock
- Hamilton
- Heidfeld
- Kovalainen
- Kubica
- Massa
- Raikkonen
- Trulli
- Vettel


In [ ]:
print(f"\nRISULTATI PILOTA: {nome.capitalize()}") #Stampa il nome del pilota con la prima lettera maiuscola
print("----------------------")
print(f"Punti totali: {risultato[0]}")
print(f"Vittorie: {risultato[1]}")
print(f"Podi: {risultato[2]}")

Funzione 2

Genera un dizionario contenente i nomi dei piloti come chiavi e il loro punteggio totale come valori. Il dizionario viene poi utilizzato per creare una classifica generale dei piloti.

Infine, la classifica sarà salvata in un file di testo (Drivers_Standings_2008.txt) con il seguente formato:

Drivers Standings 2008 Formula 1

NomePilota1: PunteggioTotale

NomePilota2: PunteggioTotale


In [ ]:
def genera_classifica(file_csv):
    #Sistema punti
    punti = {1:10, 2:8, 3:6, 4:5, 5:4, 6:3, 7:2, 8:1}

    #Dizionario vuoto che conterrà i punteggi totali dei piloti
    classifica = {}

    #Apertura del file CSV per leggere i dati delle gare
    with open(file_csv, newline='') as file:
        lettore = csv.DictReader(file)  #legge il CSV e trasforma ogni riga in un dizionario
        for riga in lettore:            #scorre il file riga per riga
            pilota = riga["Driver"].strip()     #prende il nome del pilota e toglie eventuali spazi
            posizione = int(riga["Position"])   #converte la posizione in un numero intero
            punteggio = punti.get(posizione, 0) #assegna i punti in base alla posizione

            #Aggiorna il punteggio totale del pilota (parte da 0 se non era ancora nel dizionario)
            classifica[pilota] = classifica.get(pilota, 0) + punteggio

    # Ordina i piloti per punteggio in ordine decrescente
    classifica_ordinata = dict(sorted(classifica.items(), key=lambda x: x[1], reverse=True))

    # Crea e scrive il file di testo con la classifica finale
    with open("Drivers_Standings_2008.txt", "w") as f:
        f.write("Drivers Standings 2008 Formula 1\n")  #intestazione del file
        for nome, tot in classifica_ordinata.items():
            f.write(f"{nome}: {tot}\n")  #scrive ogni pilota e il suo punteggio

    print("Classifica creata e salvata con successo")  #messaggio di conferma

    # Ritorna la classifica ordinata come dizionario
    return classifica_ordinata

In [ ]:
#Esegue la funzione per generare la classifica piloti
genera_classifica(file_csv)

In [ ]:
#Apre e mostra a schermo il contenuto del file di testo appena creato
with open("Drivers_Standings_2008.txt", "r") as f:
    print("\n--- CLASSIFICA PILOTI ---")  #titolo della classifica in console
    print(f.read())                       #stampa il contenuto del file


Funzione 3

Crea un dizionario con i nomi dei team/costruttori come chiavi e il loro punteggio totale come valori. Il punteggio di ciascun team è la somma dei punti ottenuti dai piloti che hanno gareggiato per quel costruttore.

Questa funzione utilizza i dati precedentemente generati per i piloti e calcola la classifica dei costruttori. Anche questa informazione è essenziale per avere una visione chiara delle prestazioni dei team durante l'anno.

In [ ]:
def classifica_costruttori(file_csv):
    #Sistema punti
    punti = {1:10, 2:8, 3:6, 4:5, 5:4, 6:3, 7:2, 8:1}

    #Dizionario per accumulare i punti totali dei team/costruttori
    costruttori = {}

    #Apre il file CSV e legge i dati
    with open(file_csv, newline='') as file:
        lettore = csv.DictReader(file)  #legge il file CSV e trasforma ogni riga in un dizionario (colonne → valori)
        for riga in lettore:            #scorre ogni riga del file
            team = riga["Team"].strip()           #prende il nome del team e rimuove eventuali spazi
            posizione = int(riga["Position"])     #converte la posizione del pilota in un numero intero
            punteggio = punti.get(posizione, 0)   #ottiene i punti per quella posizione (0 se oltre l’8° posto)

            #Aggiunge i punti del pilota al totale del suo team
            costruttori[team] = costruttori.get(team, 0) + punteggio

    #Ordina i team per punteggio totale (dal più alto al più basso)
    classifica_ordinata = dict(sorted(costruttori.items(), key=lambda x: x[1], reverse=True))

    #Crea e scrive il file di testo con la classifica finale dei costruttori
    with open("Constructors_Standings_2008.txt", "w") as f:
        f.write("Constructors Standings 2008 Formula 1\n")  #intestazione del file
        for team, tot in classifica_ordinata.items():
            f.write(f"{team}: {tot}\n")  #scrive ogni team e il suo punteggio

    #Messaggio di conferma
    print("Classifica costruttori creata con successo!")

    #Restituisce la classifica ordinata come dizionario
    return classifica_ordinata


In [ ]:
#Esegue la funzione
classifica_costruttori(file_csv)

In [ ]:
#Apre e mostra a schermo il contenuto del file di testo appena creato
with open("Constructors_Standings_2008.txt", "r") as f:
    print("\n--- CLASSIFICA COSTRUTTORI ---")  #titolo della classifica
    print(f.read())                            #stampa il contenuto del file